<a href="https://colab.research.google.com/github/nathanchapero-creator/swapmeet_sales_analysis/blob/main/JewlerySalesCleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install gspread pandas

import gspread
import pandas as pd
import json
from google.colab import userdata

# 1. Pull the secret string from Colab's vault
creds_string = userdata.get('GOOGLE_CREDENTIALS')

# 2. Convert that string back into a JSON dictionary format
creds_dict = json.loads(creds_string)

# 3. Log in using the dictionary instead of a file!
gc = gspread.service_account_from_dict(creds_dict)

# Now extract your data exactly like before:
sheet = gc.open("Cypress Sales Transaction Log")
raw_tab = sheet.worksheet("Original Responses")
df_responses = pd.DataFrame(raw_tab.get_all_records())


In [7]:
# Pull Weather Data
weather_tab = sheet.worksheet("Weather Data")
df_weather = pd.DataFrame(weather_tab.get_all_records())

# Pull Gross Sales Data
gross_tab = sheet.worksheet("Gross Sales Data")
df_gross = pd.DataFrame(gross_tab.get_all_records())

In [8]:
# Normalizing Date Columns to create a derived key to join
df_responses["Merge_Date"] = pd.to_datetime(df_responses["Timestamp"]).dt.normalize()
df_weather["Date"] = pd.to_datetime(df_weather["Date"])
df_gross["Date"] = pd.to_datetime(df_gross["Date"])

In [9]:
# Organizing Sterling Silver items
df_responses["Item Category"] = df_responses["Item Category"].replace({
    'Earring, Sterling Silver' : 'Sterling Silver Earring',
    'Ring, Sterling Silver' : 'Sterling Silver Ring',
    'Necklace, Sterling Silver' : 'Sterling Silver Necklace',
    'Necklace, Earring, Sterling Silver' : 'Sterling Silver Earring, Necklace'})


In [11]:
df_responses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Timestamp               200 non-null    object        
 1   Item Category           200 non-null    object        
 2   Price Range             200 non-null    object        
 3   Items Sold              200 non-null    object        
 4   Sold at Sticker Price?  200 non-null    object        
 5   Payment Method          200 non-null    object        
 6   Repeat Customer?        200 non-null    object        
 7   Merge_Date              200 non-null    datetime64[ns]
dtypes: datetime64[ns](1), object(7)
memory usage: 12.6+ KB


In [12]:
# Merging gross sales data with weather data ()
df_gross_weather = df_gross.merge(df_weather, on=["Date"], how = "left")